임계값 조정과 ROC-AUC 곡선

- 놓치면 절단날 때: 재현율(Recall)을 높힌다 ex)암환자 판별
- 찍은건 확실할 때 : 정밀도(Precision)을 높힌다. ex) 스팸메일

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

#정답(y_true)과 예측한 확률값(y_scores)
y_true = np.array([0, 0, 1 ,1 , 0 , 1, 0, 1, 1, 0])
y_scores = np.array([0.1, 0.4, 0.35, 0.8, 0.2, 0.9, 0.15, 0.7, 0.65, 0.3])

print(f"======실제 정답:{y_true}========")
print(f"=====예측 확률값:{y_scores}=========")

# sklearn 을 이용한 ROC 곡선 좌표(FPR(x축), TPR(y축))
fpr, tpr, shresholds = roc_curve(y_true,y_scores)

#sklearn을 이용한 AUC
auc_score = roc_auc_score(y_true, y_scores)
print(f"모델의 최종 ROC-AUC:, {auc_score:.4f}")


In [ ]:
#ROC 곡선 시각화 대시보드 구축
plt.figure(figsize =(8, 6))
# 모델의 ROC 곡선 플롯
plt.plot(fpr, tpr, color = 'darkorange', lw = 2, label = f"ROC Curve(AUC = {auc_score:.4f})")

#대각선 기준선(AUC=0.5인 무작위 모델의 성능)
plt.plot([0, 1],[0, 1], color='navy', lw=2, linestyle='--',label='Random Guess(AUC = 0.5)')

plt.xlim([0.0,1.0])
plt.ylim([0.0,1.05])
plt.xlabel('False Positive Rate(GPR)', fontsize = 11)
plt.ylabel('True Positive Rate(TPR)', fontsize = 11)
plt.title('Receiver Opearating Characteristic (ROC) Curve', fontsize = 13, fontweight = 'bold')
plt.legend(loc='lower right',fontsize = 11)
plt.grid(True,color='black', linestyle='--',alpha=0.5)
plt.show()

- 주황색 곡선이 위쪽 위 구석으로 올라갈 수록 모델의 성능이 우수하다. 대각선 점보다 아래로 떨어지면 동전 던지기보다 못한 최악의 모델이다.
- 분류모델의 기본 임계값(0.5)은 불균형 데이터에서 무력하며 비지니스 목적에 맞게 조정해야 한다.
-ROC 곡선은 임계값 변화에 따른 FPR과 TPR의 관계를 보여주며, AUC점수를 통해 모델의 전반적인 변별력을 객관적으로 평가할 수 있다.

비지도 학습과 K-means 군집화
- 지도학습과 비지도 학습 비교 테이블
  - 지도학습: features(X) + Target(y) 존재
  - 비지도학습 : Feature(X)만 존재

- K-Means 군집화 : 중심점을 찾아서 떠나는 여정
  - 비슷한 데이터끼리 알아서 묶어주는 방식
  -데이터들을 총 K개의 그룹으로 묶어 주는 가장 직관적이고 널리 사용되는 알고리즘
  -K는 사용자가 직접 지정해주어야 한다.
  -동그란 구형형태의 군집은 잘 찾지만 초승달 모양처럼 복잡하게 꼬인 형태는 잘 찾지 못한다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn에서 가상의 군집데이터 생성용함수와 K-Means 임포트
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

X, y_true = make_blobs( #군집화용 가상 데이터 생성 함수(X(피처데이터) 2차원 배열, y(정답레벨)을 반환해준다.)
    n_samples=300,      #데이터의 갯수
    n_features=2,       #차원 수(2차원)
    centers=3,          #군집덩어리 수
    cluster_std=0.60,   #군집 데이트들의 표준편차
    random_state=42
)
df = pd.DataFrame(X, columns = ['Feature_1', 'Feature_2'])
print(df.head())

# K-Means 모델 생성 및 학습
kmeans = KMeans(n_clusters=3, random_state = 42, n_init=10)
# 정답은 넘겨주지않고 오직 데이터만 학습시킨다.
kmeans.fit(X)
# 모델이 예측한 각 데이터별 군집 번호 가져오기
df['Cluster'] = kmeans.labels_ #사이킷런에서 _로 끝나는 속성은 학습 뒤에 생성되는 결과물을 나타낸다.
#알고리즘이 찾아낸 최종 3개 중심점 좌표 추출
print("=====[K-Means 가 스스로 찾아난 3개의 중심점 좌표]=====")
centroids = kmeans.cluster_centers_
print(centroids)
#군집 결과 시각화
#y_kmeans = kmeans.predict(X)


In [ ]:
!apt-get install -y fonts-nanum
!fc-cache -fv

In [ ]:
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 군집화 결과 및 중심점 시각화
plt.figure(figsize=(10, 6))

#데이터 포인트들을 군집별로 색상을 다르게하여 산점도로 표현
sns.scatterplot(         # seaborn 산점도 그래프 함수 (하이레벨)
    x='Feature_1',
    y='Feature_2',
    hue='Cluster',      # 카테고리별 색상이 달라짐
    data=df,            # 데이터프레임 지정
    s=50,               # 산점도 크기
    palette='viridis',
    alpha=0.8)
# K-Means가 찾아낸 중심점을 눈에 띄는 빨간색의 'X' 마커로 표시
plt.scatter(              # matplotlib 산점도 그래프 함수 (로우레벨)
    centroids[:,0],       # 모든 중심좌표의 0변열을 가져온다.(X좌표)
    centroids[:,1],       # 모든 중심좌표의 1번열을 가져온다. (y좌표)
    marker='X',
    s=200,
    color='red',
    label='Centroids(중심점)'
)
plt.title("K-means Clustering Results(K=3)", fontsize=14, fontweight='bold')
plt.xlabel("Feature 1", fontsize=12)
plt.ylabel("Feature 2", fontsize=12)
plt.legend()
plt.grid(True, color ='black', linestyle='--', alpha=0.7)
plt.show()